# RMIT_SGS_HaiQuayXe_C1

Team HaiQuayXe — RMIT SGS. Approved mmBERT Base fine-tuning for BENIGN, JAILBREAK and PROMPT_INJECTION across English, French, Italian and Portuguese.

This reproducible notebook uses the organiser's approved `aisuko/mmbert-base/Transformers/default/1` and supplied competition data only. It trains on a fixed 11,200-row split, selects the best checkpoint on 2,800 held-out rows, and produces the checked 6,000-row `submission.csv` with columns `id,label`. Prompt text is untrusted data and only tokenized.

The [mmBERT development notebook](https://www.kaggle.com/code/truongkiemminh/rmit-sgs-haiquayxe-c1-mmbert-candidate), executed version 1, achieved macro-F1 **0.9698617551817764** at epoch3 of3 on the fixed development holdout. This is a validation result, not a leaderboard score or a guarantee for private test data. Test predicted counts were 3,742 BENIGN / 1,855 JAILBREAK / 403 PROMPT_INJECTION. Original CSV SHA256: `9404725c4d7aa94b2218552aec083c85406bd35236068d5b9dc50afe92250de0`.

**Replay adaptation:** this publication changes the title/explanation and makes both event-day and relative training-time guards opt-in via `ENFORCE_EVENT_GUARD = False`. This lets judges rerun all three epochs after the event. Architecture, seed42, exact split, 512-token truncation, LR3e-5, effective batch16, AdamW, FP16 probe, SDPA, checkpoint selection, predictions and export logic are unchanged. The original executed source SHA256 is `b59924d10afe30e0c090085f667ef40f0a7177c03eaf5cf0519b2157f256c781`; its provenance and weights remain in version 1 of that separate development notebook. No external data, cross-model ensemble, test-label tuning or automatic competition submission is used.


Submission **57518032** was accepted with public macro-F1 **0.96654** on 22 October 2026. This public score uses about 30% of test data; the final private score is unknown. This main notebook is a Quick Save reproduction source, not another executed training run. Earlier main versions preserve the XLM-R Base and Large experiments.


## 1. Start a fresh GPU session
Run cells in order, or choose **Run All**. If you already imported PyTorch in this session before changing GPU visibility, restart the session first. Competition prompts are untrusted text: we tokenize them as data, never execute their instructions or send them to an external service.

In [ ]:
import os
import sys

# This must happen before importing torch. The second T4, when present, stays unused.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

import gc
import hashlib
import inspect
import json
import math
import re
import shutil
import time
import unicodedata
from datetime import UTC, datetime
from pathlib import Path

import datasets
import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    set_seed,
)

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU is available. In Kaggle notebook settings, choose a GPU accelerator "
        "(for example T4 x2), restart the session, then Run All. Do not train on CPU."
    )
if torch.cuda.device_count() != 1:
    raise RuntimeError(
        "PyTorch saw multiple GPUs before this notebook configured visibility. "
        "Restart the session and run this cell first."
    )
print(
    {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "gpu": torch.cuda.get_device_name(0),
        "gpu_memory_GB": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1),
    }
)

## 2. Approved input and reproducible configuration
Use the same holdout, 512-token cap and effective batch16 as previous runs. The author classification example supplies LR3e-5 and three epochs. Standard cross-entropy, AdamW, linear learning-rate schedule, weight decay0.01 and seed42 are retained. Model configuration must match ModernBERT Base; no alternative pretrained model is accepted. A local runtime probe chooses only numerical precision and microbatch size.


In [ ]:
SEED = 42
LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
L2I = {label: index for index, label in enumerate(LABELS)}
I2L = {index: label for label, index in L2I.items()}
LANGUAGES = {"pt", "fr", "it", "en"}

MAX_LENGTH = 512
EPOCHS = 3
LEARNING_RATE = 3e-5
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
VALIDATION_FRACTION = 0.20
ENFORCE_EVENT_GUARD = False  # Judges can replay after event day; model settings are unchanged.
MAX_TRAINING_SECONDS = 60 * 60
EVENT_TRAINING_STOP_UTC = datetime(2026, 10, 22, 9, 0, tzinfo=UTC)
now_utc = datetime.now(UTC)
ACTIVE_TRAINING_STOP_UTC = (
    EVENT_TRAINING_STOP_UTC
    if ENFORCE_EVENT_GUARD and now_utc.date() == EVENT_TRAINING_STOP_UTC.date()
    else None
)
if ACTIVE_TRAINING_STOP_UTC is not None and now_utc >= ACTIVE_TRAINING_STOP_UTC:
    raise RuntimeError("The event-day training cutoff has passed. Preserve the accepted candidate.")
if torch.cuda.get_device_properties(0).total_memory < 14 * 2**30:
    raise RuntimeError("This full-fine-tuning candidate requires at least 14 GiB of GPU memory.")

DATA_PATH = "/kaggle/input/competitions/rmit-hackathon-2026"
MODEL_PATH = "/kaggle/input/models/aisuko/mmbert-base/transformers/default/1"
WORK = Path("/kaggle/working/haiquayxe_c1")
WORK.mkdir(parents=True, exist_ok=True)
set_seed(SEED)


def locate_data(preferred):
    required = {"train.csv", "test.csv", "sample_submission.csv"}
    candidates = [Path(preferred), Path("/kaggle/input/rmit-hackathon-2026")]
    for path in candidates:
        if all((path / name).is_file() for name in required):
            return path
    found = sorted(
        {
            path.parent
            for path in Path("/kaggle/input").rglob("train.csv")
            if all((path.parent / name).is_file() for name in required)
        }
    )
    if len(found) != 1:
        raise FileNotFoundError(
            "Attach the RMIT Hackathon 2026 competition data. If multiple datasets "
            f"match, set DATA_PATH explicitly. Matching directories: {found}"
        )
    return found[0]


def locate_model(preferred):
    preferred = Path(preferred)
    if (preferred / "config.json").is_file():
        return preferred
    found = []
    for config_path in Path("/kaggle/input").rglob("config.json"):
        if "mmbert-base" not in str(config_path).lower():
            continue
        try:
            config = json.loads(config_path.read_text(encoding="utf-8"))
        except (OSError, ValueError):
            continue
        if config.get("model_type") == "modernbert":
            found.append(config_path.parent)
    if len(found) != 1:
        raise FileNotFoundError(
            "Attach the organiser's mmBERT Base model input. "
            f"Set MODEL_PATH explicitly if needed. Matching directories: {found}"
        )
    return found[0]


DATA = locate_data(DATA_PATH)
MODEL = locate_model(MODEL_PATH)
backbone_config = AutoConfig.from_pretrained(MODEL, local_files_only=True, trust_remote_code=False)
reference_compile_supported = hasattr(backbone_config, "reference_compile")
if reference_compile_supported:
    backbone_config.reference_compile = False
print(
    "reference_compile config supported:", reference_compile_supported, "; torch.compile disabled"
)
if (
    backbone_config.model_type != "modernbert"
    or backbone_config.hidden_size != 768
    or backbone_config.num_hidden_layers != 22
    or backbone_config.vocab_size != 256000
):
    raise ValueError("This notebook expects the approved mmBERT Base checkpoint.")
print("Data:", DATA)
print("Approved model:", MODEL)
print("Effective training batch:", TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION)

## 3. Check the data and create an honest validation split
The split preserves the language/class mix. Normalized duplicates, if present, stay on the same side of the split. A grouped split is approximate, so the actual counts are always printed. Normalization is used only to detect duplicates; the original text is used as model input. These checks do not print prompt contents.

In [ ]:
train = pd.read_csv(DATA / "train.csv", dtype=str, keep_default_na=False)
test = pd.read_csv(DATA / "test.csv", dtype=str, keep_default_na=False)
sample = pd.read_csv(DATA / "sample_submission.csv", dtype=str, keep_default_na=False)


def require(condition, message):
    if not condition:
        raise ValueError(message)


require(set(train.columns) == {"id", "text", "language", "label"}, "Unexpected train schema.")
require(set(test.columns) == {"id", "text", "language"}, "Unexpected test schema.")
require(list(sample.columns) == ["id", "label"], "Sample columns must be id,label.")
require(len(train) == 14000 and len(test) == 6000, "Expected 14,000 train and 6,000 test rows.")
require(len(sample) == len(test), "Sample row count does not match test.")
for frame_name, frame in [("train", train), ("test", test), ("sample", sample)]:
    require(frame["id"].str.strip().ne("").all(), f"Blank ID in {frame_name}.")
    require(frame["id"].is_unique, f"Duplicate ID in {frame_name}.")
for frame_name, frame in [("train", train), ("test", test)]:
    require(frame["text"].str.strip().ne("").all(), f"Blank prompt in {frame_name}.")
    require(set(frame["language"]) <= LANGUAGES, f"Unknown language in {frame_name}.")
require(set(train["label"]) == set(LABELS), "Unexpected or missing training class.")
require(set(test["id"]) == set(sample["id"]), "Sample/test IDs do not match.")
require(set(train["id"]).isdisjoint(test["id"]), "Train and test IDs overlap.")


def text_group(text):
    normalized = unicodedata.normalize("NFKC", text).casefold()
    normalized = re.sub(r"\s+", " ", normalized).strip()
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


train["_group"] = train["text"].map(text_group)
train["_label_id"] = train["label"].map(L2I)
strata = train["label"] + "|" + train["language"]
indices = np.arange(len(train))
if train["_group"].is_unique:
    train_indices, validation_indices = train_test_split(
        indices, test_size=VALIDATION_FRACTION, stratify=strata, random_state=SEED
    )
    split_method = "label+language stratified holdout"
else:
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    train_indices, validation_indices = next(
        splitter.split(indices, strata, groups=train["_group"])
    )
    split_method = "grouped label+language stratified holdout (5 folds; first held out)"

tr = train.iloc[train_indices].reset_index(drop=True)
va = train.iloc[validation_indices].reset_index(drop=True)
require(set(tr["_group"]).isdisjoint(va["_group"]), "Duplicate text leaked into validation.")
require(set(tr["label"]) == set(LABELS), "Training split is missing a class.")
require(set(va["label"]) == set(LABELS), "Validation split is missing a class.")
print("Loaded all training rows:", len(train))
print("Split:", split_method)
print("Fit rows:", len(tr), "Validation rows:", len(va), "Test rows:", len(test))
print("Normalized duplicate training rows:", int(train["_group"].duplicated().sum()))
conflicts = train.groupby("_group")["label"].nunique().gt(1).sum()
print("Normalized duplicate groups with conflicting labels:", int(conflicts))
print(
    pd.concat(
        [
            tr.groupby(["language", "label"]).size().rename("fit"),
            va.groupby(["language", "label"]).size().rename("validation"),
        ],
        axis=1,
    )
    .fillna(0)
    .astype(int)
    .to_string()
)

split_manifest = pd.concat(
    [
        tr[["id", "language", "label"]].assign(split="fit"),
        va[["id", "language", "label"]].assign(split="validation"),
    ],
    ignore_index=True,
)
split_manifest.to_csv(WORK / "split_manifest.csv", index=False)

# Fail closed if a library/runtime change produces a different holdout/order.
EXPECTED_FIT_ID_SHA256 = "ebcb437a2991f571f1152f4329cb82e8800ee19aee33fe63c9c7985788cc1eef"
EXPECTED_VALIDATION_ID_SHA256 = "afee70977c27d3d759e54561d31973f6b8df1f87c796a9d1690b3a733832db49"
fit_id_sha256 = hashlib.sha256("\n".join(tr["id"]).encode("utf-8")).hexdigest()
validation_id_sha256 = hashlib.sha256("\n".join(va["id"]).encode("utf-8")).hexdigest()
require(fit_id_sha256 == EXPECTED_FIT_ID_SHA256, "Fit IDs differ from v2; comparison is invalid.")
require(validation_id_sha256 == EXPECTED_VALIDATION_ID_SHA256, "Validation IDs differ from v2.")
print("Exact v2 fit and validation IDs/order verified by SHA-256.")

## 4. Tokenize
Retain v2's original first-512-token representation and dynamic padding. A seeded sample of 1,000 training prompts reports truncation only; it does not change the chosen representation.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL, local_files_only=True, use_fast=True, trust_remote_code=False
)
length_sample = train["text"].sample(n=min(1000, len(train)), random_state=SEED).tolist()
length_encoding = tokenizer(
    length_sample,
    truncation=False,
    padding=False,
    return_attention_mask=False,
    return_token_type_ids=False,
    verbose=False,
)
lengths = np.array([len(tokens) for tokens in length_encoding["input_ids"]])
truncation_diagnostics = {
    "sample_size": len(length_sample),
    "max_length": MAX_LENGTH,
    "fraction_longer_than_max_length": float((lengths > MAX_LENGTH).mean()),
    "median_tokens": float(np.median(lengths)),
    "p95_tokens": float(np.quantile(lengths, 0.95)),
}
print("Training-only token-length sample:", truncation_diagnostics)
del length_encoding, length_sample


def make_dataset(frame, labelled=True):
    values = {"text": frame["text"].tolist()}
    if labelled:
        values["labels"] = frame["label"].map(L2I).astype(int).tolist()
    dataset = Dataset.from_dict(values)
    return dataset.map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH),
        batched=True,
        remove_columns=["text"],
        load_from_cache_file=False,
    )


ds_tr = make_dataset(tr)
ds_va = make_dataset(va)
ds_te = make_dataset(test, labelled=False)
collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
print("Tokenized rows (fit / validation / test):", len(ds_tr), len(ds_va), len(ds_te))

## 5. T4 compatibility and numerical probe
Load standard ModernBERT in FP32 parameter storage and force `attn_implementation="sdpa"`. The classification head is trainable. Disable optional reference compilation and use gradient checkpointing. Probe the longest training microbatch without parameter updates, checking loss/logits/gradients for finite values. Prefer FP16 autocast when valid; otherwise retry FP32 and record the reason. On CUDA OOM only, reduce microbatch4→2 and increase accumulation to8. A second OOM or non-finite FP32 result fails clearly. Optimizer state is measured during actual training, not claimed from a forward/backward probe alone.


In [ ]:
set_seed(SEED)
model_config = backbone_config
model_config.num_labels = len(LABELS)
model_config.id2label = I2L
model_config.label2id = L2I
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    config=model_config,
    local_files_only=True,
    trust_remote_code=False,
    dtype=torch.float32,
    attn_implementation="sdpa",
)
if hasattr(model.config, "reference_compile"):
    model.config.reference_compile = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.to("cuda")
model.train()
parameter_count = sum(p.numel() for p in model.parameters())
print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad),
    flush=True,
)
print("Attention backend:", model.config._attn_implementation, flush=True)
require(model.config._attn_implementation == "sdpa", "Explicit SDPA backend was not applied.")
longest = sorted(range(len(ds_tr)), key=lambda i: len(ds_tr[i]["input_ids"]), reverse=True)[:4]
probe_history = []


def smoke_probe(batch_size, fp16):
    set_seed(SEED)
    model.zero_grad(set_to_none=True)
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    batch = None
    output = None
    try:
        batch = {
            key: value.to("cuda")
            for key, value in collator([ds_tr[i] for i in longest[:batch_size]]).items()
        }
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=fp16):
            output = model(**batch)
        finite_loss = bool(torch.isfinite(output.loss).item())
        finite_logits = bool(torch.isfinite(output.logits).all().item())
        finite_gradients = False
        if finite_loss and finite_logits:
            output.loss.backward()
            finite_gradients = all(
                bool(torch.isfinite(p.grad).all().item())
                for p in model.parameters()
                if p.grad is not None
            )
        result = {
            "precision": "fp16_autocast" if fp16 else "fp32",
            "batch_size": batch_size,
            "finite_loss": finite_loss,
            "finite_logits": finite_logits,
            "finite_gradients": finite_gradients,
            "initial_loss": float(output.loss.item()) if finite_loss else None,
            "peak_allocated_GiB": torch.cuda.max_memory_allocated() / 2**30,
            "oom": False,
        }
        result["passed"] = finite_loss and finite_logits and finite_gradients
        return result
    except torch.cuda.OutOfMemoryError:
        return {
            "precision": "fp16_autocast" if fp16 else "fp32",
            "batch_size": batch_size,
            "oom": True,
            "passed": False,
        }
    finally:
        model.zero_grad(set_to_none=True)
        del batch, output
        gc.collect()
        torch.cuda.empty_cache()


USE_FP16 = True
precision_fallback_reason = None
while True:
    probe = smoke_probe(TRAIN_BATCH_SIZE, USE_FP16)
    probe_history.append(probe)
    print("NUMERICAL_PROBE:", json.dumps(probe), flush=True)
    if probe["passed"]:
        break
    if probe["oom"]:
        if TRAIN_BATCH_SIZE == 4:
            TRAIN_BATCH_SIZE = 2
            EVAL_BATCH_SIZE = 2
            GRADIENT_ACCUMULATION = 8
            continue
        raise RuntimeError(
            "ModernBERT probe exceeded GPU memory even at microbatch2; do not silently change model/optimizer."
        )
    if USE_FP16:
        USE_FP16 = False
        precision_fallback_reason = (
            "non-finite FP16 loss, logits or gradients in training-only smoke probe"
        )
        continue
    raise RuntimeError(
        "ModernBERT FP32 probe produced non-finite outputs/gradients; refusing training."
    )

require(TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION == 16, "Effective training batch changed.")
print("Verified training precision:", "fp16" if USE_FP16 else "fp32", flush=True)
print("Verified microbatch/accumulation:", TRAIN_BATCH_SIZE, GRADIENT_ACCUMULATION, flush=True)
print(
    "Smoke excludes AdamW optimizer-state allocation; real step10 records that memory.", flush=True
)
(WORK / "numerical_probe.json").write_text(json.dumps(probe_history, indent=2), encoding="utf-8")
set_seed(SEED)

## 6. Fine-tune and select by validation macro-F1
Train up to three complete epochs using the same AdamW, standard cross-entropy, learning rate, weight decay and schedule as v2. Evaluate each epoch and restore the highest validation macro-F1 checkpoint. A callback measures actual first-epoch runtime and stops if the planned run cannot fit its budget; it preserves and clearly labels an early checkpoint. The safety guard may also stop and evaluate/save at an optimizer-step boundary when its hard limit is reached. Actual epochs, reasons and projection are exported. No competition submission occurs.


In [ ]:
class TrainingDeadlineGuard(TrainerCallback):
    """Keep real trained/evaluated checkpoints when event-time budget is exhausted."""

    def __init__(self, report_path):
        self.report_path = report_path
        self.started = None
        self.reason = None
        self.first_epoch_projection_seconds = None
        self.events = []

    def _record(self, event, state, **details):
        payload = {
            "event": event,
            "epoch": float(state.epoch or 0),
            "step": int(state.global_step),
            "elapsed_seconds": time.monotonic() - self.started if self.started else 0,
            "utc": datetime.now(UTC).isoformat(),
            **details,
        }
        self.events.append(payload)
        self.report_path.write_text(json.dumps(self.events, indent=2), encoding="utf-8")
        print("TRAINING_BUDGET:", json.dumps(payload), flush=True)

    def on_train_begin(self, args, state, control, **kwargs):
        self.started = time.monotonic()
        torch.cuda.reset_peak_memory_stats()
        self._record(
            "started",
            state,
            max_training_seconds=MAX_TRAINING_SECONDS if ENFORCE_EVENT_GUARD else None,
            event_guard_enabled=ENFORCE_EVENT_GUARD,
            event_cutoff_utc=ACTIVE_TRAINING_STOP_UTC.isoformat()
            if ACTIVE_TRAINING_STOP_UTC
            else None,
        )

    def on_step_end(self, args, state, control, **kwargs):
        elapsed = time.monotonic() - self.started
        now = datetime.now(UTC)
        if state.global_step == 10:
            self._record(
                "optimizer_memory_checked",
                state,
                allocated_GiB=torch.cuda.memory_allocated() / 2**30,
                peak_allocated_GiB=torch.cuda.max_memory_allocated() / 2**30,
                reserved_GiB=torch.cuda.memory_reserved() / 2**30,
            )
        if ENFORCE_EVENT_GUARD and self.reason is None:
            if elapsed >= MAX_TRAINING_SECONDS:
                self.reason = "relative_training_budget_exhausted"
            elif ACTIVE_TRAINING_STOP_UTC and now >= ACTIVE_TRAINING_STOP_UTC:
                self.reason = "event_day_training_cutoff_reached"
            if self.reason:
                self._record(self.reason, state)
                control.should_training_stop = True
                control.should_evaluate = True
                control.should_save = True
        return control

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and ("loss" in logs or "eval_loss" in logs):
            print(
                "TRAINING_PROGRESS:",
                json.dumps({"step": state.global_step, "epoch": state.epoch, **logs}, default=str),
                flush=True,
            )
        return control

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        self._record(
            "epoch_evaluated", state, validation_macro_f1=(metrics or {}).get("eval_macro_f1")
        )
        if state.epoch and state.epoch >= 1 and self.first_epoch_projection_seconds is None:
            elapsed = time.monotonic() - self.started
            # The first measured epoch includes evaluation; allow five minutes extra
            # for later checkpoint overhead. Do not restart or change quality settings.
            projected_total = elapsed / float(state.epoch) * EPOCHS + 300
            projected_remaining = max(0, projected_total - elapsed)
            self.first_epoch_projection_seconds = projected_total
            available_event_seconds = (
                (ACTIVE_TRAINING_STOP_UTC - datetime.now(UTC)).total_seconds()
                if ACTIVE_TRAINING_STOP_UTC
                else float("inf")
            )
            self._record(
                "first_epoch_measured",
                state,
                projected_total_seconds=projected_total,
                projected_remaining_seconds=projected_remaining,
                available_event_seconds=available_event_seconds
                if math.isfinite(available_event_seconds)
                else None,
                validation_macro_f1=(metrics or {}).get("eval_macro_f1"),
            )
            if (
                ENFORCE_EVENT_GUARD
                and self.reason is None
                and (
                    projected_total > MAX_TRAINING_SECONDS
                    or projected_remaining > available_event_seconds
                )
            ):
                self.reason = "first_epoch_projection_exceeds_training_budget"
                self._record(self.reason, state)
                control.should_training_stop = True
                control.should_save = True
        return control

    def on_train_end(self, args, state, control, **kwargs):
        self._record(
            "finished",
            state,
            stop_reason=self.reason,
            planned_epochs=EPOCHS,
            actual_epochs=float(state.epoch or 0),
        )
        return control


deadline_guard = TrainingDeadlineGuard(WORK / "training_budget.json")


def compute_metrics(prediction):
    logits = prediction.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = np.argmax(logits, axis=-1)
    return {
        "macro_f1": f1_score(
            prediction.label_ids,
            predictions,
            labels=list(range(len(LABELS))),
            average="macro",
            zero_division=0,
        )
    }


batches_per_epoch = math.ceil(len(ds_tr) / TRAIN_BATCH_SIZE)
optimizer_steps = math.ceil(batches_per_epoch / GRADIENT_ACCUMULATION) * EPOCHS
warmup_steps = max(1, math.ceil(optimizer_steps * 0.06))

argument_values = dict(
    output_dir=str(WORK / "checkpoints"),
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    lr_scheduler_type="linear",
    optim="adamw_torch",
    fp16=USE_FP16,
    torch_compile=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=False,
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=2,
    save_only_model=True,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=0,
    eval_accumulation_steps=1,
)
# Kaggle images can carry different supported Transformers versions.
argument_parameters = inspect.signature(TrainingArguments.__init__).parameters
strategy_parameter = (
    "eval_strategy" if "eval_strategy" in argument_parameters else "evaluation_strategy"
)
argument_values[strategy_parameter] = "epoch"
training_args = TrainingArguments(**argument_values)
trainer_values = dict(
    model=model,
    args=training_args,
    train_dataset=ds_tr,
    eval_dataset=ds_va,
    data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[deadline_guard],
)
trainer_parameters = inspect.signature(Trainer.__init__).parameters
if "processing_class" in trainer_parameters:
    trainer_values["processing_class"] = tokenizer
elif "tokenizer" in trainer_parameters:
    trainer_values["tokenizer"] = tokenizer
trainer = Trainer(**trainer_values)
started = time.monotonic()
train_result = trainer.train()
training_seconds = time.monotonic() - started
print("Training minutes:", round(training_seconds / 60, 1))
print("Actual epochs:", trainer.state.epoch, "Planned epochs:", EPOCHS)
print("Budget stop reason:", deadline_guard.reason)
print("Best validation macro-F1:", trainer.state.best_metric)
print("Best checkpoint:", trainer.state.best_model_checkpoint)

## 7. Evaluate the selected checkpoint and save the model
Inspect rare-class `PROMPT_INJECTION` F1 as well as overall macro-F1. Language scores can reveal uneven performance. This is an internal validation score, not the Kaggle leaderboard score. Repeatedly tuning on one holdout can overfit it, so make a few deliberate comparisons rather than many tiny changes.

The confusion matrix uses true labels for rows and predicted labels for columns. Model files, configuration, split IDs, and validation predictions make the result reproducible. Competition text is not copied to these reports.

In [ ]:
validation_output = trainer.predict(ds_va, metric_key_prefix="validation")
validation_logits = validation_output.predictions
if isinstance(validation_logits, tuple):
    validation_logits = validation_logits[0]
validation_logits = np.asarray(validation_logits)
require(np.isfinite(validation_logits).all(), "Non-finite validation logits.")
validation_prediction = validation_logits.argmax(axis=-1)
validation_truth = va["_label_id"].to_numpy()
validation_macro_f1 = f1_score(
    validation_truth,
    validation_prediction,
    labels=list(range(len(LABELS))),
    average="macro",
    zero_division=0,
)
report = classification_report(
    validation_truth,
    validation_prediction,
    labels=list(range(len(LABELS))),
    target_names=LABELS,
    zero_division=0,
    output_dict=True,
)
print("Held-out macro-F1:", round(validation_macro_f1, 6))
print(
    classification_report(
        validation_truth,
        validation_prediction,
        labels=list(range(len(LABELS))),
        target_names=LABELS,
        zero_division=0,
        digits=4,
    )
)
confusion = confusion_matrix(
    validation_truth, validation_prediction, labels=list(range(len(LABELS)))
)
print("Confusion matrix (rows=true, columns=predicted):")
print(pd.DataFrame(confusion, index=LABELS, columns=LABELS).to_string())
language_scores = {}
for language in sorted(LANGUAGES):
    mask = va["language"].eq(language).to_numpy()
    if mask.any():
        language_scores[language] = {
            "rows": int(mask.sum()),
            "macro_f1": float(
                f1_score(
                    validation_truth[mask],
                    validation_prediction[mask],
                    labels=list(range(len(LABELS))),
                    average="macro",
                    zero_division=0,
                )
            ),
        }
print("Per-language:", json.dumps(language_scores, indent=2))

validation_rows = va[["id", "language", "label"]].rename(columns={"label": "true_label"}).copy()
validation_rows["predicted_label"] = [I2L[int(index)] for index in validation_prediction]
for index, label in enumerate(LABELS):
    validation_rows[f"logit_{label}"] = validation_logits[:, index]
validation_rows.to_csv(WORK / "validation_predictions.csv", index=False)
np.save(WORK / "validation_logits.npy", validation_logits)

run_config = {
    "approved_backbone": "jhu-clsp/mmBERT-base",
    "experiment": "approved mmBERT-base: 512 tokens, 3 planned epochs, LR3e-5, effective batch16",
    "attention_backend": model.config._attn_implementation,
    "torch_compile": False,
    "reference_compile_supported": reference_compile_supported,
    "precision": "fp16" if USE_FP16 else "fp32",
    "precision_fallback_reason": precision_fallback_reason,
    "numerical_probe_history": probe_history,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "learning_rate_reference": "author mmBERT classification example (3e-5)",
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "optimizer": "adamw_torch",
    "microbatch_numeric_difference_from_v2": "same4x4 normally;2x8 only if needed for GPU memory",
    "actual_epochs_completed": float(trainer.state.epoch or 0),
    "training_stopped_early": float(trainer.state.epoch or 0) < EPOCHS - 1e-6,
    "training_budget_stop_reason": deadline_guard.reason,
    "first_epoch_projected_seconds": deadline_guard.first_epoch_projection_seconds,
    "training_budget_seconds": MAX_TRAINING_SECONDS if ENFORCE_EVENT_GUARD else None,
    "event_guard_enabled": ENFORCE_EVENT_GUARD,
    "event_training_cutoff_utc": ACTIVE_TRAINING_STOP_UTC.isoformat()
    if ACTIVE_TRAINING_STOP_UTC
    else None,
    "fit_id_sha256": fit_id_sha256,
    "validation_id_sha256": validation_id_sha256,
    "parameter_count": parameter_count,
    "peak_training_allocated_GiB": torch.cuda.max_memory_allocated() / 2**30,
    "checkpoint_policy": "model-only checkpoints; remove duplicates after best-model and history export",
    "attached_model_path": str(MODEL),
    "seed": SEED,
    "labels": LABELS,
    "max_length": MAX_LENGTH,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation": GRADIENT_ACCUMULATION,
    "split_method": split_method,
    "fit_rows": len(tr),
    "validation_rows": len(va),
    "training_seconds": training_seconds,
    "best_epoch": next(
        (
            entry.get("epoch")
            for entry in trainer.state.log_history
            if entry.get("eval_macro_f1") == trainer.state.best_metric
        ),
        None,
    ),
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "versions": {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
    },
    "token_length_diagnostics": truncation_diagnostics,
}
validation_summary = {
    "macro_f1": float(validation_macro_f1),
    "classification_report": report,
    "confusion_matrix": confusion.tolist(),
    "per_language": language_scores,
}
(WORK / "run_config.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")
(WORK / "validation_metrics.json").write_text(
    json.dumps(validation_summary, indent=2), encoding="utf-8"
)
trainer.save_model(str(WORK / "best_model"))
tokenizer.save_pretrained(WORK / "best_model")
trainer.save_state()

# Preserve the chosen model and full training history before deleting redundant
# checkpoints generated by this run. No user inputs or unrelated files are removed.
best_dir = (WORK / "best_model").resolve()
checkpoint_dir = (WORK / "checkpoints").resolve()
expected_work = Path("/kaggle/working/haiquayxe_c1").resolve()
require(WORK.resolve() == expected_work, "Unexpected output root; refusing checkpoint cleanup.")
require(
    checkpoint_dir.parent == expected_work and checkpoint_dir.name == "checkpoints",
    "Unexpected checkpoint directory; refusing cleanup.",
)
require(
    best_dir.parent == expected_work and best_dir.name == "best_model",
    "Unexpected model export directory; refusing cleanup.",
)
weight_files = list(best_dir.glob("*.safetensors")) + list(best_dir.glob("pytorch_model*.bin"))
require((best_dir / "config.json").is_file(), "Best-model config was not exported.")
require(
    any(path.is_file() and path.stat().st_size > 0 for path in weight_files),
    "Best-model weights were not exported; preserving checkpoints.",
)
history_source = checkpoint_dir / "trainer_state.json"
history = json.loads(history_source.read_text(encoding="utf-8"))
require(bool(history.get("log_history")), "Training history is missing; preserving checkpoints.")
shutil.copy2(history_source, WORK / "trainer_state.json")
require(
    json.loads((WORK / "trainer_state.json").read_text(encoding="utf-8")) == history,
    "Training history export did not round-trip; preserving checkpoints.",
)
shutil.rmtree(checkpoint_dir)
print("Preserved best_model and trainer_state.json; removed redundant generated checkpoints.")
print("Saved selected model and evaluation reports in:", WORK)

## 8. Predict test rows and create the checked submission
Predictions are joined by ID into the sample's exact row order. The notebook rereads the output and checks the schema, row count, ID coverage, and labels. It does not consume a Kaggle submission attempt automatically.

In [ ]:
test_output = trainer.predict(ds_te, metric_key_prefix="test")
test_logits = test_output.predictions
if isinstance(test_logits, tuple):
    test_logits = test_logits[0]
test_logits = np.asarray(test_logits)
require(test_logits.shape == (len(test), len(LABELS)), "Wrong test prediction shape.")
require(np.isfinite(test_logits).all(), "Non-finite test logits.")
test_prediction = test_logits.argmax(axis=-1)
predictions = test[["id"]].copy()
predictions["label"] = [I2L[int(index)] for index in test_prediction]
require(predictions["id"].is_unique, "Duplicate prediction IDs.")
submission = sample[["id"]].merge(
    predictions, on="id", how="left", validate="one_to_one", sort=False
)
require(submission["label"].notna().all(), "Missing prediction after joining IDs.")
require(submission["id"].tolist() == sample["id"].tolist(), "Sample ID order changed.")
require(set(submission["label"]) <= set(LABELS), "Prediction contains an invalid label.")
submission_path = Path("/kaggle/working/submission.csv")
submission.to_csv(submission_path, index=False)
written = pd.read_csv(submission_path, dtype=str, keep_default_na=False)
require(list(written.columns) == ["id", "label"], "Written CSV has incorrect headers.")
require(len(written) == 6000 and written["id"].is_unique, "Written CSV has incorrect rows or IDs.")
require(written["id"].tolist() == sample["id"].tolist(), "Written CSV has incorrect ID order.")
require(set(written["label"]) <= set(LABELS), "Written CSV has invalid labels.")
require(written.equals(submission), "Written CSV did not round-trip exactly.")

np.save(WORK / "test_logits.npy", test_logits)
test_rows = predictions.copy()
for index, label in enumerate(LABELS):
    test_rows[f"logit_{label}"] = test_logits[:, index]
test_rows.to_csv(WORK / "test_predictions.csv", index=False)
print("READY:", submission_path)
print("Rows:", len(written), "Columns:", list(written.columns))
print("Predicted class counts (not a quality score):")
print(written["label"].value_counts().reindex(LABELS, fill_value=0).to_string())
print("Held-out macro-F1:", round(validation_macro_f1, 6))

## 9. Review before spending a submission
Verify the same holdout hashes, numerical precision chosen, actual epochs and all-class/per-language metrics. Compare with XLM-R Base v2 (.942501) and Base-long (.949179) on those exact rows. This independent architecture experiment also changes learning rate to the author's recommendation; attribution to a single change is not claimed.

An accepted public score is available from earlier XLM-R runs. Do not replace it or combine pretrained backbones automatically. If mmBERT earns a substantive validation improvement, choose its settings before any all-data refit and preserve time for final output checking and notebook delivery. No submission or external publication occurs in this notebook.
